# 05 -- MFE/MAE exit analysis

Paired script: `analysis/calculate_mfe_mae.py`. Per-trade maximum favorable/adverse
excursion from bar high/low data, using `trade_math.compute_r_multiple` -- kept
algebraically identical to `ExitManager.mqh`'s `EM_ComputeR` (TASK-030).

**Uses clearly-labelled SYNTHETIC bar data.** Real-data run: PENDING.

In [ ]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.calculate_mfe_mae import run

In [ ]:
# Same fixture hand-verified in tests/test_calculate_mfe_mae.py: a long trade,
# entry 100, stop 98 -> MFE 5 (2.5R), MAE 3 (-1.5R).
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_mfemae_demo_"))

pd.DataFrame(
    {
        "symbol": ["XAUUSD"] * 4,
        "timestamp": pd.date_range("2026-07-21T00:00:00Z", periods=4, freq="h"),
        "high": [101.0, 105.0, 103.0, 104.0],
        "low": [99.0, 100.0, 97.0, 101.0],
    }
).to_csv(tmp_dir / "bars.csv", index=False)

pd.DataFrame(
    [
        {
            "trade_id": "t1",
            "symbol": "XAUUSD",
            "is_long": "True",
            "entry_time": "2026-07-21T00:00:00Z",
            "exit_time": "2026-07-21T03:00:00Z",
            "entry_price": 100.0,
            "stop_price": 98.0,
        }
    ]
).to_csv(tmp_dir / "trades.csv", index=False)

In [ ]:
result = run(
    tmp_dir / "trades.csv",
    tmp_dir / "bars.csv",
    output_csv=tmp_dir / "mfe_mae.csv",
    errors_json=tmp_dir / "errors.json",
    repo_path=PROJECT_ROOT.parents[1],
)

r = result.results[0]
print(f"mfe_price = {r.mfe_price:.2f}  mfe_r = {r.mfe_r:.2f}")
print(f"mae_price = {r.mae_price:.2f}  mae_r = {r.mae_r:.2f}")

assert abs(r.mfe_price - 5.0) < 1e-9 and abs(r.mfe_r - 2.5) < 1e-9
assert abs(r.mae_price - 3.0) < 1e-9 and abs(r.mae_r - (-1.5)) < 1e-9

## Real-data run: PENDING

Requires real per-trade OHLC bar coverage -- no real trade history exists yet.